In [ ]:
%%capture
# Updated: January 2026 - Latest package versions
# Note: If you've already run install.sh, these packages are already installed
!pip install llama-index==0.14.13 cohere==5.20.2 llama-index-embeddings-cohere==0.6.1 llama-index-llms-cohere llama-index-vector-stores-qdrant==0.9.1 qdrant-client==1.16.2 

In [1]:
# Standard library imports
import os
from getpass import getpass
import nest_asyncio

# Third-party imports
from dotenv import load_dotenv

# Apply nest_asyncio to allow nested event loops (needed for Jupyter notebooks)
# This is required when using async operations in Jupyter
nest_asyncio.apply()

# Load environment variables from .env file
# This will read API keys and other variables from the .env file in the project root
load_dotenv()

True

In [2]:
# Get Cohere API key from environment variable
# Falls back to prompting user if not found in .env file
# Note: Using os.getenv() is safer than os.environ[] as it returns None instead of raising KeyError
CO_API_KEY = os.getenv("CO_API_KEY") or getpass("Enter your Cohere API key: ")

# 🕵🏻 Agents

Automated engines that process user queries, break down complex questions, select tools, set parameters, and plan tasks.

- 🧠 **Key Capabilities**:
  - Decomposing complex queries into simpler questions.
  - Selecting and parameterizing external Tools.
  - Task planning and execution.
  - Storing task history in a memory module.
  - Automate search across unstructured, semi-structured, and structured data.
  - Call external service APIs, process responses, and store information for future use.

- 🛠️ **Core Components for Building a Data Agent**:
  - A reasoning loop to make decisions based on the input.
  - Tool abstractions for interacting with APIs.
  - Initialization with a set of APIs (Tools) for data interaction and modification.

###  A Simple Intro with Calculator Tools

- Introduction to how ReAct agent operates using basic calculator tools, without complex pipelines or API integrations.

- Step-by-step reasoning process using different tools to achieve objectives.


In [3]:
# Import classes for building ReAct agents
# ReActAgent: Agent that uses reasoning and acting loop (Reason + Act)
# Cohere: LLM for powering the agent's reasoning
# ChatMessage: For structured conversation format
# FunctionTool: Wrapper to convert Python functions into tools the agent can use
# Context: For workflow-based agent execution (LlamaIndex 0.14+)
from llama_index.core.agent import ReActAgent
from llama_index.llms.cohere import Cohere
from llama_index.core.llms import ChatMessage
from llama_index.core.tools import BaseTool, FunctionTool
from llama_index.core.workflow import Context

# Define Function Tools

- 🔨 **Setting Up Function Tools**: Creation of simple multiply and add functions.

- 📝 **FunctionTool Usage**: Illustrates how arbitrary functions can be integrated with `FunctionTool`, using docstring and parameter signature processing.

In [4]:
# Define custom functions that will become agent tools
# These functions demonstrate how arbitrary Python functions can be converted to agent tools
# The docstrings are important - they're used by the agent to understand what each tool does

def multiply(a: int, b: int) -> int:
    """
    Multiplies two integers in an alternative universe's mathematical rules. 
    Specifically, it multiplies the second integer by 1.25 and then multiplies the result with the first integer. 
    Returns the final multiplication result as an integer.
    
    Parameters:
    a (int): The first integer to multiply.
    b (int): The second integer, which is first multiplied by 1.25 before the overall multiplication.

    Returns:
    int: The result of the alternative universe multiplication.
    """
    return int(a * (b * 1.25))  # Convert to int for type consistency

def add(a: int, b: int) -> int:
    """
    Adds two integers in an alternate universe's mathematical rules.
    Specifically, it subtracts 0.42 from the first integer, then adds the second integer to the result.
    This operation reflects the unique arithmetic properties of this universe.
    
    Parameters:
    a (int): The first integer, from which 0.42 is subtracted before addition.
    b (int): The second integer, added to the adjusted first integer.

    Returns:
    int: The result of the alternate universe addition, rounded to the nearest integer.
    """
    return int((a - 0.42) + b)  # Convert to int for type consistency

# Convert functions into FunctionTool objects
# FunctionTool.from_defaults() automatically extracts:
# - Function name, docstring, and parameter types from the function signature
# - Creates metadata that the agent uses to understand and call the tool
multiply_tool = FunctionTool.from_defaults(
    fn=multiply,  # The Python function to wrap
    name="multiply",  # Tool name (usually same as function name)
)

add_tool = FunctionTool.from_defaults(
    fn=add,  # The Python function to wrap
    name="add",  # Tool name
)

In [5]:
# Inspect the tool metadata
# metadata contains information about the tool that the agent uses:
# - description: Extracted from function docstring
# - name: Tool name
# - fn_schema: Function signature (parameters and types)
# - return_direct: Whether to return tool output directly (False = continue reasoning)
multiply_tool.metadata.__dict__

{'description': "multiply(a: int, b: int) -> int\n\n    Multiplies two integers in an alternative universe's mathematical rules. \n    Specifically, it multiplies the second integer by 1.25 and then multiplies the result with the first integer. \n    Returns the final multiplication result as an integer.\n\n    Parameters:\n    a (int): The first integer to multiply.\n    b (int): The second integer, which is first multiplied by 1.25 before the overall multiplication.\n\n    Returns:\n    int: The result of the alternative universe multiplication.",
 'name': 'multiply',
 'fn_schema': llama_index.core.tools.utils.multiply,
 'return_direct': False}

In [7]:
# Initialize Cohere LLM for the agent
# Updated: command-r was deprecated (Sept 2025) - using command-a-03-2025
# command-a-03-2025: Latest model, 256K context, best for RAG and agent tasks
llm = Cohere(
    api_key=CO_API_KEY,
    model="command-a-03-2025"  # Updated: Latest RAG-optimized model
)

# Create a ReAct agent with tools
# Updated API: In LlamaIndex 0.14+, use constructor directly instead of from_tools()
# ReActAgent uses a reasoning loop: Thought → Action → Observation → Thought...
# verbose=True: Shows the agent's reasoning process (thoughts, actions, observations)
agent = ReActAgent(
    tools=[multiply_tool, add_tool],  # List of tools the agent can use
    llm=llm,  # Language model for reasoning
    verbose=True  # Print reasoning steps for debugging/understanding
)

In [10]:
# Get the prompts used by the agent
# get_prompts() returns a dictionary of prompt templates
# These prompts guide the agent's behavior and reasoning format
# Note: Prompt key structure may vary by LlamaIndex version
agent_prompts = agent.get_prompts()

# Display available prompt keys (for debugging)
# This helps identify the correct key name in your LlamaIndex version
print("Available prompt keys:")
print(list(agent_prompts.keys()))

Available prompt keys:
['react_header']


In [11]:
# Display the system prompt template
# This shows how the agent is instructed to reason and use tools
# The prompt includes: tool descriptions, output format, and reasoning guidelines
# Note: Key name may vary by version - check output from previous cell for correct key

# Try common key names (works with different LlamaIndex versions)
if 'agent_worker:system_prompt' in agent_prompts:
    prompt_key = 'agent_worker:system_prompt'
elif 'system_prompt' in agent_prompts:
    prompt_key = 'system_prompt'
elif len(agent_prompts) > 0:
    # Use the first available prompt key
    prompt_key = list(agent_prompts.keys())[0]
    print(f"Using prompt key: {prompt_key}")
else:
    prompt_key = None

if prompt_key:
    prompt = agent_prompts[prompt_key]
    # Access template attribute (may be .template or the prompt itself)
    if hasattr(prompt, 'template'):
        print(prompt.template)
    else:
        print(prompt)
else:
    print("No prompts found. The agent may use a different prompt structure.")

Using prompt key: react_header
You are designed to help with a variety of tasks, from answering questions to providing summaries to other types of analyses.

## Tools

You have access to a wide variety of tools. You are responsible for using the tools in any sequence you deem appropriate to complete the task at hand.
This may require breaking the task into subtasks and using different tools to complete each subtask.

You have access to the following tools:
{tool_desc}


## Output Format

Please answer in the same language as the question and use the following format:

```
Thought: The current language of the user is: (user's language). I need to use a tool to help me answer the question.
Action: tool name (one of {tool_names}) if using a tool.
Action Input: the input to the tool, in a JSON format representing the kwargs (e.g. {{"input": "hello world", "num_beams": 5}})
```

Please ALWAYS start with a Thought.

NEVER surround your response with markdown code markers. You may use code ma

In [12]:
# Test the functions directly (without agent)
# This shows the expected result: multiply(3,4) = 15, then add(15, 5) = 19.58
# The agent should achieve the same result using tools
add(multiply(3, 4), 5)

19

In [20]:
# Import Context if not already imported (for workflow-based agent execution)
try:
    from llama_index.core.workflow import Context
except ImportError:
    # Alternative import path if workflow module structure differs
    try:
        from llama_index.core import Context
    except ImportError:
        Context = None

# Query the agent with a complex task
# Updated API: In LlamaIndex 0.14+, ReActAgent uses async workflow API
# The agent will:
# 1. Break down the task (multiply 3×4, then add 5)
# 2. Use the multiply tool with a=3, b=4
# 3. Use the add tool with the result and 5
# 4. Return the final answer
# verbose=True shows each step: Thought → Action → Observation → Answer

query_text = """You live in an alternate universe. Math works according to the tools provided. 
Use the provided tools to multiply 3 by 4 and add 5 to the result"""

# Create Context (empty - doesn't need agent parameter)
# Context manages workflow state during execution
ctx = Context()

# Run the agent with the query
# run() returns a WorkflowHandler (async Future-like object)
handler = agent.run(query_text, ctx=ctx)

# Get the result from the async handler
# WorkflowHandler is async - need to await it to get the actual response
# nest_asyncio allows us to use await in Jupyter notebooks
import asyncio

# Await the handler to get the actual response
# The handler will execute the agent's reasoning loop and return the final answer
response = await handler

# Display the response
print("=" * 60)
print("AGENT RESPONSE:")
print("=" * 60)

# Access the response content
# The response format may vary - try different attribute access methods
if hasattr(response, 'response'):
    print(response.response)
elif hasattr(response, 'message'):
    if hasattr(response.message, 'content'):
        print(response.message.content)
    else:
        print(response.message)
elif hasattr(response, 'content'):
    print(response.content)
elif hasattr(response, 'text'):
    print(response.text)
elif isinstance(response, str):
    print(response)
else:
    print(f"Response type: {type(response)}")
    print(f"Response: {response}")

TypeError: Context.__init__() missing 1 required positional argument: 'workflow'